# Basic Session Analysis

Loads `session_*.csv`, `audio_events_*.csv`, and `shot_labels_*.csv` from a single session,
aligns labeled shots to the nearest heartbeat, and visualises HR around hits vs misses.

**Run this notebook from the `capstone/` root directory.**

In [ ]:
from pathlib import Path
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor':   '#050505',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#f2e4b3',
    'xtick.color':      '#f2e4b3',
    'ytick.color':      '#f2e4b3',
    'text.color':       '#f2e4b3',
    'grid.color':       '#2a2a2a',
    'figure.dpi':       120,
})
GOLD = '#CFB53B'
GREEN = '#2ca02c'
GRAY  = '#888888'
RED   = '#d62728'

## 1. Select session files

Auto-picks the most recent complete session (one that has all three file types).
Override `SESSION_PREFIX` to load a specific session.

In [ ]:
LOG_DIR = Path('../logs')          # relative to notebooks/

# ── auto-detect most recent session with all three file types ────────────────
session_files  = sorted(LOG_DIR.glob('session_*.csv'),      key=lambda p: p.stem)
audio_files    = sorted(LOG_DIR.glob('audio_events_*.csv'), key=lambda p: p.stem)
label_files    = sorted(LOG_DIR.glob('shot_labels_*.csv'),  key=lambda p: p.stem)

# Build a lookup keyed by (device_addr, timestamp) suffix
def _key(path, prefix):
    return path.stem[len(prefix):]

session_keys = {_key(f, 'session_'): f      for f in session_files}
audio_keys   = {_key(f, 'audio_events_'): f for f in audio_files}
label_keys   = {_key(f, 'shot_labels_'): f  for f in label_files}

# Find keys present in session + at least one of audio/labels
available = [(k, session_keys[k]) for k in sorted(session_keys, reverse=True)
             if k in audio_keys or k in label_keys]

if not available:
    # Fall back: any session file
    available = [(k, v) for k, v in sorted(session_keys.items(), reverse=True)]

if not available:
    raise FileNotFoundError(f'No session files found in {LOG_DIR.resolve()}')

SESSION_KEY    = available[0][0]
SESSION_FILE   = session_keys[SESSION_KEY]
AUDIO_FILE     = audio_keys.get(SESSION_KEY)
LABELS_FILE    = label_keys.get(SESSION_KEY)

print('Session:',  SESSION_FILE.name)
print('Audio:',    AUDIO_FILE.name  if AUDIO_FILE  else '(none)')
print('Labels:',   LABELS_FILE.name if LABELS_FILE else '(none)')

## 2. Load CSVs

In [ ]:
session_df = pd.read_csv(SESSION_FILE)
session_df.dropna(subset=['elapsed_sec', 'rr_ms', 'inst_hr_bpm'], inplace=True)
session_df.sort_values('elapsed_sec', inplace=True)
session_df.reset_index(drop=True, inplace=True)

print(f'Session rows: {len(session_df)}')
if len(session_df) > 0:
    dur = session_df['elapsed_sec'].max() - session_df['elapsed_sec'].min()
    print(f'Duration: {dur:.1f} s  ({dur/60:.1f} min)')
    print(f'HR range: {session_df["inst_hr_bpm"].min():.0f} – {session_df["inst_hr_bpm"].max():.0f} bpm')
    display(session_df.head(4))
else:
    print('  (empty session — no beats were logged)')

In [ ]:
if AUDIO_FILE:
    audio_df = pd.read_csv(AUDIO_FILE)
    audio_df.sort_values('elapsed_sec', inplace=True)
    audio_df.reset_index(drop=True, inplace=True)
    print(f'Audio events: {len(audio_df)}')
    if len(audio_df):
        print(f'dB range: {audio_df["audio_db"].min():.1f} – {audio_df["audio_db"].max():.1f}')
        display(audio_df.head(4))
else:
    audio_df = pd.DataFrame(columns=['elapsed_sec', 'audio_db', 'timestamp_epoch'])
    print('No audio events file.')

In [ ]:
if LABELS_FILE:
    labels_df = pd.read_csv(LABELS_FILE)
    labels_df.sort_values('shot_time_elapsed', inplace=True)
    labels_df.reset_index(drop=True, inplace=True)
    print(f'Shot labels: {len(labels_df)}')
    if len(labels_df):
        display(labels_df)
else:
    labels_df = pd.DataFrame(columns=['shot_time_elapsed', 'label', 'inst_hr_bpm', 'audio_db_peak', 'db_threshold'])
    print('No shot labels file.')

## 3. Full-session HR + audio overview

In [ ]:
if len(session_df) == 0:
    print('No session data to plot.')
else:
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    # HR trace
    ax = axes[0]
    ax.plot(session_df['elapsed_sec'], session_df['inst_hr_bpm'],
            color=GOLD, linewidth=1.2, label='HR')
    ax.set_ylabel('HR (BPM)')
    ax.grid(True, alpha=0.3)

    # Shot markers on HR plot
    for _, row in labels_df.iterrows():
        c = GREEN if row['label'] == 'hit' else GRAY
        ax.axvline(row['shot_time_elapsed'], color=c, linewidth=1, linestyle='--', alpha=0.8)

    # Audio dB trace
    ax2 = axes[1]
    if len(audio_df):
        ax2.scatter(audio_df['elapsed_sec'], audio_df['audio_db'],
                    s=6, color='#4da6ff', alpha=0.7, label='audio events')
    ax2.set_xlabel('Elapsed (s)')
    ax2.set_ylabel('dB')
    ax2.grid(True, alpha=0.3)

    # Threshold line
    if len(labels_df) and 'db_threshold' in labels_df.columns:
        thr = labels_df['db_threshold'].iloc[0]
        ax2.axhline(thr, color=GOLD, linewidth=1, linestyle='--', label=f'threshold {thr:.0f} dB')
        ax2.legend(fontsize=8)

    hit_patch  = mpatches.Patch(color=GREEN, label='hit')
    miss_patch = mpatches.Patch(color=GRAY,  label='miss')
    axes[0].legend(handles=[hit_patch, miss_patch], fontsize=8, loc='upper right')

    fig.suptitle('Session Overview', color=GOLD, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

## 4. Align labeled shots to nearest beat

In [ ]:
def nearest_beat(shot_time, session_df):
    """Return the session row whose elapsed_sec is closest to shot_time."""
    if len(session_df) == 0:
        return None
    idx = (session_df['elapsed_sec'] - shot_time).abs().idxmin()
    return session_df.loc[idx]

if len(labels_df) and len(session_df):
    aligned = []
    for _, shot in labels_df.iterrows():
        beat = nearest_beat(shot['shot_time_elapsed'], session_df)
        if beat is not None:
            aligned.append({
                'shot_time':   shot['shot_time_elapsed'],
                'label':       shot['label'],
                'beat_time':   beat['elapsed_sec'],
                'inst_hr_bpm': beat['inst_hr_bpm'],
                'rr_ms':       beat['rr_ms'],
                'audio_peak':  beat['audio_db_peak'],
                'offset_s':    abs(shot['shot_time_elapsed'] - beat['elapsed_sec']),
            })
    aligned_df = pd.DataFrame(aligned)
    print(f'Aligned {len(aligned_df)} shots to beats')
    display(aligned_df)
else:
    aligned_df = pd.DataFrame()
    print('No labels or session data to align.')

## 5. HR window around each shot  (± 10 seconds)

In [ ]:
HALF_WINDOW = 10.0  # seconds before/after each labeled shot

if len(labels_df) == 0 or len(session_df) == 0:
    print('No labeled shots or session data available.')
else:
    hits  = labels_df[labels_df['label'] == 'hit']['shot_time_elapsed'].tolist()
    misses = labels_df[labels_df['label'] == 'miss']['shot_time_elapsed'].tolist()

    n_shots = len(labels_df)
    ncols = min(n_shots, 3)
    nrows = (n_shots + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5 * ncols, 3.5 * nrows),
                             sharey=True)
    axes = np.array(axes).flatten()

    for ax_idx, (_, shot) in enumerate(labels_df.iterrows()):
        t0 = shot['shot_time_elapsed']
        label = shot['label']
        color = GREEN if label == 'hit' else GRAY

        mask = (
            (session_df['elapsed_sec'] >= t0 - HALF_WINDOW) &
            (session_df['elapsed_sec'] <= t0 + HALF_WINDOW)
        )
        window = session_df[mask]
        rel_time = window['elapsed_sec'] - t0

        ax = axes[ax_idx]
        if len(window) > 0:
            ax.plot(rel_time, window['inst_hr_bpm'], color=GOLD, linewidth=1.5)
        ax.axvline(0, color=color, linewidth=1.5, linestyle='--', label=label)
        ax.set_title(f'Shot {ax_idx + 1}: {label.upper()}',
                     color=color, fontsize=10)
        ax.set_xlabel('Time relative to shot (s)', fontsize=8)
        ax.set_ylabel('BPM', fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-HALF_WINDOW, HALF_WINDOW)

    # Hide unused axes
    for ax in axes[n_shots:]:
        ax.set_visible(False)

    fig.suptitle(f'HR ± {HALF_WINDOW:.0f}s around each labeled shot', color=GOLD, fontsize=12)
    plt.tight_layout()
    plt.show()

## 6. Average HR at shot: hits vs misses

In [ ]:
if len(aligned_df) == 0:
    print('No aligned shots to compare.')
else:
    by_label = aligned_df.groupby('label')['inst_hr_bpm'].agg(['mean', 'std', 'count'])
    print('\nHR at shot time (bpm):')
    display(by_label)

    fig, ax = plt.subplots(figsize=(5, 4))
    labels_order = [l for l in ['hit', 'miss'] if l in by_label.index]
    colors_order = [GREEN if l == 'hit' else GRAY for l in labels_order]
    means = [by_label.loc[l, 'mean'] for l in labels_order]
    stds  = [by_label.loc[l, 'std'] if not np.isnan(by_label.loc[l, 'std']) else 0
             for l in labels_order]

    bars = ax.bar(labels_order, means, color=colors_order, yerr=stds,
                  capsize=6, edgecolor='#333', linewidth=0.8)
    ax.set_ylabel('HR at shot (BPM)')
    ax.set_title('Average HR at shot time', color=GOLD)
    ax.grid(True, axis='y', alpha=0.3)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, mean + 1,
                f'{mean:.0f}', ha='center', va='bottom', fontsize=10, color='#f2e4b3')
    plt.tight_layout()
    plt.show()

## 7. Summary statistics

In [ ]:
print('=== Session summary ===')

if len(session_df):
    print(f'  Beats logged:   {len(session_df)}')
    print(f'  Duration:       {session_df["elapsed_sec"].max():.1f} s')
    print(f'  HR mean:        {session_df["inst_hr_bpm"].mean():.1f} bpm')
    print(f'  HR max:         {session_df["inst_hr_bpm"].max():.1f} bpm')
    print(f'  HR min:         {session_df["inst_hr_bpm"].min():.1f} bpm')
    over = session_df['audio_over_threshold'].sum()
    print(f'  Beats w/ audio spike: {int(over)}')

if len(audio_df):
    print(f'  Audio events:   {len(audio_df)}')
    print(f'  Audio dB mean:  {audio_df["audio_db"].mean():.1f}')
    print(f'  Audio dB max:   {audio_df["audio_db"].max():.1f}')

if len(labels_df):
    hits  = (labels_df['label'] == 'hit').sum()
    misses = (labels_df['label'] == 'miss').sum()
    total = len(labels_df)
    print(f'  Shots labeled:  {total}  ({hits} hits, {misses} misses)')
    print(f'  Hit rate:       {100*hits/total:.0f}%' if total else '')
    if 'inst_hr_bpm' in aligned_df.columns and len(aligned_df):
        print(f'  Avg HR at shot: {aligned_df["inst_hr_bpm"].mean():.1f} bpm')

print('Done.')